In [23]:
from fastapi import FastAPI, Request, Form
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
import pandas as pd
import matplotlib.pyplot as plt
from io import BytesIO
import base64
from datetime import datetime
from typing import Optional
import os
import nest_asyncio
import uvicorn
from threading import Thread
from dateutil.relativedelta import relativedelta

# using nest_asyncio for Jupyter compatibility
nest_asyncio.apply()
app = FastAPI()

os.makedirs("static", exist_ok=True)
app.mount("/static", StaticFiles(directory="static"), name="static")

templates = Jinja2Templates(directory=".")

df = pd.read_csv("predicted_year.csv")
df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = df['datetime'].dt.date
df['month_year'] = df['datetime'].dt.to_period('M').astype(str)

# Getting unique values for dropdowns
unique_years = sorted(df['datetime'].dt.year.unique())
unique_month_years = sorted(df['datetime'].dt.strftime('%m/%Y').unique())
unique_days = sorted(df['datetime'].dt.strftime('%d/%m/%Y').unique())

def create_dual_plots(data, title, x_column='datetime'):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # Line plot
    ax1.plot(data[x_column], data['Power Generated'], color='green', linewidth=2)
    ax1.set_title(f'{title} - Line Chart', pad=20)
    ax1.set_ylabel('Power Generated (kW)')
    ax1.grid(color='gray', linestyle=':', alpha=0.5)
    
    # Bar plot
    ax2.bar(data[x_column], data['Power Generated'], color='green', width=0.8)
    ax2.set_title(f'{title} - Bar Chart', pad=20)
    ax2.set_ylabel('Power Generated (kW)')
    ax2.grid(color='gray', linestyle=':', alpha=0.5)
    
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    
    # Save to buffer
    buf = BytesIO()
    plt.savefig(buf, format='png', facecolor='white', bbox_inches='tight', dpi=100)
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>Solar Generation Dashboard</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f5f5f5;
            margin: 0;
            padding: 20px;
            display: flex;
            justify-content: center;
            min-height: 100vh;
        }
        .main-container {
            max-width: 1000px;
            width: 100%;
        }
        .container {
            background-color: white;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            padding: 30px;
            width: 100%;
        }
        h1 {
            color: #2e7d32;
            text-align: center;
            margin-bottom: 30px;
        }
        .form-container {
            background-color: #f9f9f9;
            border-radius: 8px;
            padding: 25px;
            margin-bottom: 30px;
            border: 1px solid #e0e0e0;
        }
        .form-group {
            margin-bottom: 20px;
        }
        label {
            display: block;
            margin-bottom: 8px;
            font-weight: bold;
            color: #333;
        }
        select {
            width: 100%;
            padding: 10px;
            border: 1px solid #ddd;
            border-radius: 4px;
            background-color: white;
            font-size: 16px;
        }
        button {
            background-color: #2e7d32;
            color: white;
            border: none;
            padding: 12px 20px;
            border-radius: 4px;
            cursor: pointer;
            font-size: 16px;
            width: 100%;
            margin-top: 10px;
            transition: background-color 0.3s;
        }
        button:hover {
            background-color: #1b5e20;
        }
        .results {
            margin-top: 30px;
            text-align: center;
        }
        .plot-container {
            margin: 0 auto;
            max-width: 900px;
            text-align: center;
        }
        .plot-container img {
            max-width: 100%;
            height: auto;
            display: block;
            margin: 0 auto;
        }
        .error {
            color: #d32f2f;
            text-align: center;
            margin: 15px 0;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            margin: 20px auto;
            max-width: 900px;
        }
        th {
            background-color: #2e7d32;
            color: white;
            padding: 12px;
            text-align: left;
        }
        td {
            padding: 10px;
            border-bottom: 1px solid #ddd;
        }
        .total-power {
            font-size: 1.2em;
            font-weight: bold;
            color: #2e7d32;
            margin: 20px 0;
            text-align: center;
        }
        .range-inputs {
            display: flex;
            gap: 15px;
        }
        .range-input {
            flex: 1;
        }
        @media (max-width: 768px) {
            .range-inputs {
                flex-direction: column;
                gap: 10px;
            }
            .container {
                padding: 15px;
            }
        }
    </style>
</head>
<body>
    <div class="main-container">
        <div class="container">
            <h1>Solar Generation Prediction Dashboard</h1>
            
            <div class="form-container">
                <form method="post">
                    <div class="form-group">
                        <label>Query Type:</label>
                        <select name="query_type" id="query_type" onchange="updateForm()">
                            <option value="single_day">Single Day</option>
                            <option value="single_month">Single Month</option>
                            <option value="date_range">Date Range</option>
                        </select>
                    </div>
                    
                    <div id="single_day_input" class="form-group">
                        <label>Select Date:</label>
                        <select name="single_day">
                            {% for day in unique_days %}
                            <option value="{{ day }}">{{ day }}</option>
                            {% endfor %}
                        </select>
                    </div>
                    
                    <div id="date_range_input" class="form-group" style="display: none;">
                        <label>Date Range:</label>
                        <div class="range-inputs">
                            <div class="range-input">
                                <label>From:</label>
                                <select name="start_date">
                                    {% for day in unique_days %}
                                    <option value="{{ day }}">{{ day }}</option>
                                    {% endfor %}
                                </select>
                            </div>
                            <div class="range-input">
                                <label>To:</label>
                                <select name="end_date">
                                    {% for day in unique_days %}
                                    <option value="{{ day }}">{{ day }}</option>
                                    {% endfor %}
                                </select>
                            </div>
                        </div>
                    </div>
                    
                    <div id="month_range_input" class="form-group" style="display: none;">
                        <label>Month Range:</label>
                        <div class="range-inputs">
                            <div class="range-input">
                                <label>From:</label>
                                <select name="start_month">
                                    {% for month_year in unique_month_years %}
                                    <option value="{{ month_year }}">{{ month_year }}</option>
                                    {% endfor %}
                                </select>
                            </div>
                            <div class="range-input">
                                <label>To:</label>
                                <select name="end_month">
                                    {% for month_year in unique_month_years %}
                                    <option value="{{ month_year }}">{{ month_year }}</option>
                                    {% endfor %}
                                </select>
                            </div>
                        </div>
                    </div>
                    
                    <div id="single_month_input" class="form-group" style="display: none;">
                        <label>Select Month:</label>
                        <select name="single_month">
                            {% for month_year in unique_month_years %}
                            <option value="{{ month_year }}">{{ month_year }}</option>
                            {% endfor %}
                        </select>
                    </div>
                    
                    <div id="single_year_input" class="form-group" style="display: none;">
                        <label>Select Year:</label>
                        <select name="single_year">
                            {% for year in unique_years %}
                            <option value="{{ year }}">{{ year }}</option>
                            {% endfor %}
                        </select>
                    </div>
                    
                    <button type="submit">Get Predictions</button>
                </form>
            </div>
            
            {% if error %}
            <div class="error">{{ error }}</div>
            {% endif %}
            
            {% if plot_image %}
            <div class="results">
                <h2>{{ result_title }}</h2>
                {% if total_power %}
                <div class="total-power">
                    Total Predicted Power: {{ total_power }} kW
                </div>
                {% endif %}
                
                <div class="plot-container">
                    <img src="data:image/png;base64,{{ plot_image }}" alt="Solar Generation Plot">
                </div>
                
                {% if data_table %}
                <h3>Detailed Data</h3>
                <div style="overflow-x: auto;">
                    {{ data_table|safe }}
                </div>
                {% endif %}
            </div>
            {% endif %}
        </div>
    </div>
    
    <script>
        function updateForm() {
            const queryType = document.getElementById('query_type').value;
            const inputs = [
                'single_day_input',
                'date_range_input',
                'month_range_input',
                'single_month_input',
                'single_year_input'
            ];
            
            // Hide all inputs first
            inputs.forEach(id => {
                document.getElementById(id).style.display = 'none';
            });
            
            // Show the selected input
            document.getElementById(`${queryType}_input`).style.display = 'block';
        }
        
        // Initialize form on load
        window.onload = updateForm;
    </script>
</body>
</html>
"""
@app.get("/", response_class=HTMLResponse)
async def read_root(request: Request):
    return templates.TemplateResponse("index.html", {
        "request": request,
        "unique_years": unique_years,
        "unique_month_years": unique_month_years,
        "unique_days": unique_days
    })

@app.post("/", response_class=HTMLResponse)
async def process_query(
    request: Request,
    query_type: str = Form(...),
    single_day: Optional[str] = Form(None),
    start_date: Optional[str] = Form(None),
    end_date: Optional[str] = Form(None),
    start_month: Optional[str] = Form(None),
    end_month: Optional[str] = Form(None),
    single_month: Optional[str] = Form(None),
    single_year: Optional[str] = Form(None)
):
    error = None
    plot_image = None
    result_title = ""
    data_table = None
    total_power = None
    
    try:
        if query_type == "single_day" and single_day:
            try:
                # Parse single day
                day_date = datetime.strptime(single_day, "%d/%m/%Y").date()
                result = df[df['date'] == day_date].sort_values('datetime')
                
                if not result.empty:
                    result_title = f"Solar Generation on {day_date.strftime('%d %B %Y')}"
                    total_power = round(result['Power Generated'].sum(), 2)
                    plot_image = create_dual_plots(result, result_title)
                    
                    # Prepare data table
                    display_df = result.copy()
                    display_df['Power Generated'] = display_df['Power Generated'].round(2)
                    display_df['Date'] = display_df['datetime'].dt.strftime('%d %B %Y %H:%M')
                    display_df = display_df[['Date', 'Power Generated']]
                    data_table = display_df.to_html(classes="data-table", index=False)
                else:
                    error = f"No data found for {day_date.strftime('%d %B %Y')}"
            
            except ValueError:
                error = "Invalid date format"
        
        elif query_type == "date_range" and start_date and end_date:
            try:
                # Parse date range
                start_dt = datetime.strptime(start_date, "%d/%m/%Y").date()
                end_dt = datetime.strptime(end_date, "%d/%m/%Y").date()
                
                result = df[(df['date'] >= start_dt) & (df['date'] <= end_dt)]
                
                if not result.empty:
                    # Aggregate by day
                    daily_result = result.groupby('date')['Power Generated'].sum().reset_index()
                    result_title = f"Solar Generation from {format_date_range(start_date, end_date)}"
                    total_power = round(daily_result['Power Generated'].sum(), 2)
                    plot_image = create_dual_plots(daily_result, result_title, 'date')
                    
                    # Prepare data table
                    display_df = daily_result.copy()
                    display_df['Power Generated'] = display_df['Power Generated'].round(2)
                    display_df['Date'] = display_df['date'].apply(lambda x: x.strftime('%d %B %Y'))
                    display_df = display_df[['Date', 'Power Generated']]
                    data_table = display_df.to_html(classes="data-table", index=False)
                else:
                    error = f"No data found between {format_date_range(start_date, end_date)}"
            
            except ValueError:
                error = "Invalid date format"
        
        elif query_type == "month_range" and start_month and end_month:
            try:
                # Parse month range
                start = datetime.strptime(f"01/{start_month}", "%d/%m/%Y")
                end = datetime.strptime(f"01/{end_month}", "%d/%m/%Y") + relativedelta(months=1, days=-1)
                
                result = df[(df['datetime'] >= start) & (df['datetime'] <= end)]
                
                if not result.empty:
                    # Aggregate by day
                    daily_result = result.groupby('date')['Power Generated'].sum().reset_index()
                    result_title = f"Solar Generation from {start.strftime('%B %Y')} to {end.strftime('%B %Y')}"
                    total_power = round(daily_result['Power Generated'].sum(), 2)
                    plot_image = create_dual_plots(daily_result, result_title, 'date')
                    
                    # Prepare data table
                    display_df = daily_result.copy()
                    display_df['Power Generated'] = display_df['Power Generated'].round(2)
                    display_df['Date'] = display_df['date'].apply(lambda x: x.strftime('%d %B %Y'))
                    display_df = display_df[['Date', 'Power Generated']]
                    data_table = display_df.to_html(classes="data-table", index=False)
                else:
                    error = f"No data found between {start.strftime('%B %Y')} and {end.strftime('%B %Y')}"
            
            except ValueError:
                error = "Invalid month format"
        
        elif query_type == "single_month" and single_month:
            try:
                # Parse single month
                month, year = map(int, single_month.split('/'))
                start = datetime(year, month, 1)
                end = start + relativedelta(months=1, days=-1)
                
                result = df[(df['datetime'] >= start) & (df['datetime'] <= end)]
                
                if not result.empty:
                    # Aggregate by day
                    daily_result = result.groupby('date')['Power Generated'].sum().reset_index()
                    result_title = f"Solar Generation for {start.strftime('%B %Y')}"
                    total_power = round(daily_result['Power Generated'].sum(), 2)
                    plot_image = create_dual_plots(daily_result, result_title, 'date')
                    
                    # Prepare data table
                    display_df = daily_result.copy()
                    display_df['Power Generated'] = display_df['Power Generated'].round(2)
                    display_df['Date'] = display_df['date'].apply(lambda x: x.strftime('%d %B %Y'))
                    display_df = display_df[['Date', 'Power Generated']]
                    data_table = display_df.to_html(classes="data-table", index=False)
                else:
                    error = f"No data found for {start.strftime('%B %Y')}"
            
            except ValueError:
                error = "Invalid month format"
        
        elif query_type == "single_year" and single_year:
            try:
                year = int(single_year)
                result = df[df['datetime'].dt.year == year]
                
                if not result.empty:
                    # Aggregate by month
                    monthly_result = result.groupby('month_year')['Power Generated'].sum().reset_index()
                    monthly_result['month_year'] = pd.to_datetime(monthly_result['month_year'].astype(str)).dt.strftime('%B %Y')
                    result_title = f"Solar Generation for {year}"
                    total_power = round(monthly_result['Power Generated'].sum(), 2)
                    plot_image = create_dual_plots(monthly_result, result_title, 'month_year')
                    
                    # Prepare data table
                    display_df = monthly_result.copy()
                    display_df['Power Generated'] = display_df['Power Generated'].round(2)
                    display_df.columns = ['Month', 'Power Generated']
                    data_table = display_df.to_html(classes="data-table", index=False)
                else:
                    error = f"No data found for year {year}"
            
            except ValueError:
                error = "Invalid year format"
        else:
            error = "Please provide valid input for the selected query type"
    
    except Exception as e:
        error = f"An unexpected error occurred: {str(e)}"
    
    return templates.TemplateResponse("index.html", {
        "request": request,
        "unique_years": unique_years,
        "unique_month_years": unique_month_years,
        "unique_days": unique_days,
        "error": error,
        "plot_image": plot_image,
        "result_title": result_title,
        "data_table": data_table,
        "total_power": total_power
    })

# Save HTML template to file
with open("index.html", "w") as f:
    f.write(html_template)

# Run the server in a separate thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8038)

thread = Thread(target=run_server, daemon=True)
thread.start()

print("Server is running at http://localhost:8038")
print("You can stop the server by interrupting the kernel")
print(f"Dataset covers from {df['datetime'].min()} to {df['datetime'].max()}")

Server is running at http://localhost:8038
You can stop the server by interrupting the kernel
Dataset covers from 2025-05-01 00:00:00 to 2026-04-30 21:00:00


INFO:     Started server process [10156]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8038 (Press CTRL+C to quit)


INFO:     127.0.0.1:58750 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:58750 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:58751 - "POST / HTTP/1.1" 200 OK
